In [1]:
import sys
print(sys.version)

3.12.5 | packaged by conda-forge | (main, Aug  8 2024, 18:24:51) [MSC v.1940 64 bit (AMD64)]


In [2]:
import sys

# Ensure pip is used from the same Python executable as the notebook
# The '-m' flag runs pip as a module, which is the best practice.
!{sys.executable} -m pip install selenium

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: typing_extensions
    Found existing installation: typing-extensions None


error: uninstall-no-record-file

× Cannot uninstall typing-extensions None
╰─> The package's contents are unknown: no RECORD file was found for typing-extensions.

hint: You might be able to recover from this via: pip install --force-reinstall --no-deps typing-extensions==4.12.2


In [16]:
import time
import csv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service 
from selenium.common.exceptions import TimeoutException, WebDriverException, NoSuchElementException, ElementClickInterceptedException
import pandas as pd
import re

# --- Scraper Debugging Flag ---
# !!! IMPORTANT: SET THIS TO False TO SEE THE BROWSER WINDOW AND DEBUG !!!
HEADLESS_MODE = False 

# --- CRITICAL CONFIGURATION: UPDATE THESE PATHS ---
# 1. CHROME_BINARY_PATH: MUST point to the specific 'chrome.exe' executable (the browser).
# 2. CHROME_DRIVER_PATH: MUST point to the specific 'chromedriver.exe' executable (the driver).
CHROME_BINARY_PATH = r"C:\Program Files\Google\Chrome\Application\chrome.exe" 
# CHROME_BINARY_PATH = r"C:\Pricing_BL\chrome.exe" 
CHROME_DRIVER_PATH = r"C:\Pricing_BL\chromedriver.exe" # <-- CRITICAL: Ensure this is chromedriver.exe

# --- Item Configuration ---
ITEM_ID = "7026c01" 
# ITEM_ID = "x467c12"
# ITEM_ID = 3297
COLOR_ID = "11" 
ITEM_NAME = "Window 1 x 2 x 2 with Fixed Trans-Clear Glass" 

# --- URL Construction ---
TARGET_URL = f"https://www.bricklink.com/v2/catalog/catalogitem.page?P={ITEM_ID}&idColor={COLOR_ID}#T=P&C={COLOR_ID}"
# The historical sales file is now named using the ITEM_ID.
OUTPUT_HISTORY_FILE = f"{ITEM_ID}_sales_history.csv"
OUTPUT_LISTINGS_FILE = f'{ITEM_ID}_current_used_listings.csv'
OUTPUT_SUMMARY_FILE = 'sales_summary.csv'

# --- New Output Files for Available Stats ---
OUTPUT_NEW_STATS_FILE = f"{ITEM_ID}_stats_new_available.csv"
OUTPUT_USED_STATS_FILE = f"{ITEM_ID}_stats_used_available.csv"


TIMEOUT = 20 # seconds

# --- CSS/XPATH Selectors ---
PRICE_GUIDE_LINK_XPATH = "//td[text()='Price Guide']"
SALES_CONTAINER_XPATH = "//div[@id='_idPGContents']"
BOT_BLOCK_MESSAGE_XPATH = "//div[contains(@class, 'blp-block-message') or contains(text(), 'Access Denied')]"

# Summary Statistics (table[1]) - All 4 columns in the first main row (tr[1])
NEW_HISTORY_SUMMARY_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[1]/table[1]"
USED_HISTORY_SUMMARY_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[2]/table[1]"
NEW_AVAILABLE_SUMMARY_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[3]/table[1]"
USED_AVAILABLE_SUMMARY_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[4]/table[1]"

# Sales History (pcipgInnerTable in td[1] or td[2]) - Second main row (tr[2])
NEW_SALES_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[1]//table[contains(@class, 'pcipgInnerTable')]"
USED_SALES_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[2]//table[contains(@class, 'pcipgInnerTable')]"

# Current Listings (pcipgInnerTable in td[3] or td[4]) - Second main row (tr[2])
# Direct XPath to the inner table within the correct column for listings - SIMPLIFIED XPATH
NEW_LISTINGS_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[3]//table[contains(@class, 'pcipgInnerTable')]"
USED_LISTINGS_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[4]//table[contains(@class, 'pcipgInnerTable')]"

# Current Listing Rows: Tr that contains a link to a store in the first column
# This targets the rows containing the store icon/link
USED_LISTING_ROW_XPATH = ".//tr[td[1]/a[contains(@href, 'store.asp?sID')]]"


DOB_INPUT_ELEMENTS_XPATH = "//input[contains(@class, 'blp-age-gate__input-field')]"
DOB_SUBMIT_XPATH = "//button[normalize-space(.)='Submit' or normalize-space(.)='Enter' or normalize-space(.)='Continue']"
COOKIES_ACCEPT_XPATH = "//button[contains(text(), 'Accept all')]"

def handle_popups(driver):
    """Handles the age verification and cookie consent modals."""
    
    # 1. Handle Age Verification
    try:
        time.sleep(1.5)
        print("Checking for Age Verification pop-up and preparing to inject birth year '1983'...")
        
        digit_inputs = WebDriverWait(driver, 7).until(
            EC.presence_of_all_elements_located((By.XPATH, DOB_INPUT_ELEMENTS_XPATH))
        )
        
        num_fields = len(digit_inputs)

        if num_fields > 0:
            year_str = "1983"
            digits_to_use = year_str[-num_fields:]
            
            print(f"Found {len(digit_inputs)} potential digit fields. Injecting digits '{digits_to_use}'...")
            
            js_script = """
            arguments[0].value = arguments[1];
            arguments[0].dispatchEvent(new Event('focus', { bubbles: true }));
            arguments[0].dispatchEvent(new Event('input', { bubbles: true }));
            arguments[0].dispatchEvent(new Event('change', { bubbles: true }));
            arguments[0].dispatchEvent(new Event('blur', { bubbles: true }));
            """
            
            for i in range(num_fields):
                digit = digits_to_use[i]
                input_field = digit_inputs[i]
                driver.execute_script(js_script, input_field, digit)
                time.sleep(0.1)
            
            dob_submit = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, DOB_SUBMIT_XPATH))
            )
            driver.execute_script("arguments[0].click();", dob_submit)
            print("Submitted birth year.")
            time.sleep(3)
        
        else:
            print(f"Did not find any potential digit inputs. Skipping digit input.")
        
    except TimeoutException:
        print("Age verification modal not found or not needed.")
    except Exception as e:
        print(f"Error handling DOB modal: {type(e).__name__}: {e}. Continuing...")


    # 2. Handle Cookie Consent
    try:
        print("Checking for Cookie consent banner...")
        cookie_accept_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, COOKIES_ACCEPT_XPATH))
        )
        driver.execute_script("arguments[0].click();", cookie_accept_button)
        print("Accepted all cookies.")
        time.sleep(1)
    except TimeoutException:
        print("No cookie consent banner detected.")
    except Exception as e:
        print(f"Error handling cookie banner: {type(e).__name__}. Continuing...")

    print("Pop-up handling complete.")


def setup_driver():
    """Initializes and returns a Selenium WebDriver instance."""
    try:
        chrome_options = webdriver.ChromeOptions()
        
        # --- Human-like Options for Bot Detection Bypass ---
        chrome_options.add_argument("window-size=1280,800")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        # Ignores the 'Chrome is being controlled by automated software' message
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_experimental_option('useAutomationExtension', False)
        chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
        
        # Toggle headless mode based on the flag
        if HEADLESS_MODE:
            chrome_options.add_argument("--headless")
        
        if CHROME_DRIVER_PATH and CHROME_BINARY_PATH:
            chrome_options.binary_location = CHROME_BINARY_PATH
            service = Service(CHROME_DRIVER_PATH)
            driver = webdriver.Chrome(service=service, options=chrome_options)
        else:
            driver = webdriver.Chrome(options=chrome_options)
            
        driver.set_page_load_timeout(TIMEOUT)
        return driver
    except WebDriverException as e:
        print(f"Error initializing WebDriver: {e}")
        return None

def extract_price(text):
    """Cleans and extracts the numeric price value from a text string, handling various currencies."""
    if not text or '-' in text:
        return None
    
    # Remove currency symbols/text, including USD, CA, etc.
    cleaned_text = re.sub(r'(\$|€|£|AUD|CAD|US|CA)\s*', '', text, flags=re.IGNORECASE)
    cleaned_text = cleaned_text.replace('~', '').strip()
    
    try:
        match = re.search(r'\d+\.?\d*', cleaned_text)
        if match:
            return float(match.group(0))
        return None
    except ValueError:
        return None

def extract_monthly_sales_data(table_element, condition):
    """
    Extracts the detailed Qty and Price/Unit lines from the nested monthly sales table.
    """
    sales_rows = table_element.find_elements(By.TAG_NAME, "tr")
    extracted_records = []
    current_month = "N/A"
    
    # Keywords that indicate a summary line or header line, not a detailed sale line
    summary_keywords = {'Times Sold:', 'Total Lots:', 'Total Qty:', 'Min Price:', 'Avg Price:', 'Qty Avg Price:', 'Max Price:', 'Qty', 'Each'}

    for row in sales_rows:
        cells = row.find_elements(By.TAG_NAME, "td")
        row_text = [cell.text.strip() for cell in cells]
        
        if not any(row_text):
            continue

        # Rule 1: Identify the Month/Year Header (e.g., "December 2025")
        if len(cells) > 0 and 'pcipgSubHeader' in cells[0].get_attribute('class'):
            current_month = cells[0].text.strip()
            continue
        
        # Rule 2: Identify detailed sales rows (Qty and Price)
        # Detailed rows have a Qty (cols[1]) and Price (cols[2]) and cols[1] should be a number.
        if len(cells) == 3:
            qty_text = cells[1].text.strip()
            price_text = cells[2].text.strip()

            # Check if it's a valid sales record row (not a header like 'Qty' 'Each')
            if qty_text and price_text and qty_text.isdigit():
                
                price = extract_price(price_text)
                
                extracted_records.append({
                    'Condition': condition,
                    'Month': current_month,
                    'Quantity': qty_text,
                    'Price_per_unit': price,
                    'Raw_Price_Text': price_text
                })
        
        # Rule 3: Skip the summary rows that appear after each month
        elif len(cells) == 2:
            cell_0_text = cells[0].text.strip()
            if any(cell_0_text.startswith(kw) for kw in summary_keywords):
                continue

    return extracted_records


def scrape_summary_data(driver):
    """
    Extracts the summary statistics from all four columns in the first row.
    Returns: (history_stats, new_available_stats, used_available_stats)
    """
    all_stats = []
    
    # Define all 4 columns to scrape in the first row
    summary_configs = [
        ("NEW_HISTORY", NEW_HISTORY_SUMMARY_TABLE_XPATH),
        ("USED_HISTORY", USED_HISTORY_SUMMARY_TABLE_XPATH),
        ("NEW_AVAILABLE", NEW_AVAILABLE_SUMMARY_TABLE_XPATH),
        ("USED_AVAILABLE", USED_AVAILABLE_SUMMARY_TABLE_XPATH)
    ]
    
    for stat_type, xpath in summary_configs:
        print(f"\nAttempting to scrape {stat_type} Summary Data...")
        
        try:
            summary_table = WebDriverWait(driver, 5).until(
                EC.visibility_of_element_located((By.XPATH, xpath))
            )
            
            rows = summary_table.find_elements(By.TAG_NAME, "tr")
            record = {'Condition_Type': stat_type}
            
            for row in rows:
                cells = row.find_elements(By.TAG_NAME, "td")
                if len(cells) == 2:
                    key = cells[0].text.strip().replace(':', '')
                    value = cells[1].text.strip()
                    field_name = key.replace(' ', '_')
                    record[field_name] = value
            
            if len(record) > 1:
                all_stats.append(record)
                print(f"Successfully extracted {len(record)-1} fields for {stat_type} summary.")
            
        except TimeoutException:
            print(f"Timeout: {stat_type} Summary Table was not found at XPath: {xpath}. Skipping.")
        except Exception as e:
            print(f"Error during {stat_type} summary extraction: {type(e).__name__}: {e}. Skipping.")

    # Separate the results for return values
    history_stats = [s for s in all_stats if 'HISTORY' in s['Condition_Type']]
    new_available_stats = [s for s in all_stats if s['Condition_Type'] == 'NEW_AVAILABLE']
    used_available_stats = [s for s in all_stats if s['Condition_Type'] == 'USED_AVAILABLE']
    
    return history_stats, new_available_stats, used_available_stats


def extract_current_listings(driver, condition, table_xpath):
    """
    Extracts current listings from the specified table XPath, using presence check 
    and scrolling to fix visibility issues, and includes debugging prints.
    """
    listings_data = []
    print(f"\nScraping Current {condition} Listings...")
    
    try:
        current_listings_table = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, table_xpath))
        )
        
        # --- USER REQUEST 1: Print "head Currently Available spotted" for USED ---
        try:
            # Look for the header row inside the table
            header_xpath = ".//tr/td[contains(@class, 'pcipgSubHeader') and contains(b, 'Currently Available')]"
            # This line will throw NoSuchElementException if the header is not found
            current_listings_table.find_element(By.XPATH, header_xpath) 
            
            if condition == "USED": # <-- This is why it only prints for USED, IF the element is found
                print("head Currently Available spotted")
                
        except NoSuchElementException:
            # This is expected if the table is technically present but doesn't have the standard header
            pass
        
        # 2. Force scroll the table into view using JavaScript to resolve visibility
        driver.execute_script("arguments[0].scrollIntoView(true);", current_listings_table)
        time.sleep(1) # Small pause after scrolling

        # 3. Find listing rows inside the specific inner table
        # This XPath finds rows that start with a link (the store icon/link)
        listing_rows = current_listings_table.find_elements(By.XPATH, USED_LISTING_ROW_XPATH)
        
        first_row_printed = False # Flag for user request 2
        
        for row in listing_rows:
            try:
                # --- USER REQUEST 2: Print HTML of the first row ---
                if not first_row_printed:
                    print(f"HTML of first successful {condition} listing row:\n{row.get_attribute('outerHTML')}")
                    first_row_printed = True
                
                cols = row.find_elements(By.TAG_NAME, 'td')
                
                # Listing rows are expected to have exactly 3 TD elements: Store Link, Qty (colspan=2), Price
                if len(cols) == 3:
                    
                    store_link = cols[0].find_element(By.TAG_NAME, 'a')
                    
                    try:
                        # FIX: Changed By.TAG_TAG to By.TAG_NAME to correctly find the img element
                        store_img = store_link.find_element(By.TAG_NAME, 'img')
                        store_name = store_img.get_attribute('title').replace('Store: ', '').strip()
                    except NoSuchElementException:
                        # Fallback if no image title exists
                        store_name = store_link.text.strip()
                        
                    store_url = store_link.get_attribute('href')
                    
                    # Quantity is in cols[1], Price is in cols[2]
                    qty = cols[1].text.strip()
                    raw_price = cols[2].text.strip()
                    price = extract_price(raw_price)
                    
                    listings_data.append({
                        'Condition': condition,
                        'Store_Name': store_name,
                        'Quantity': qty,
                        'Price_per_unit': price,
                        'Raw_Price_Text': raw_price,
                        'Store_URL': store_url
                    })
                    
            except Exception as e:
                # print(f"Skipping row due to error: {e}") # Debugging skipped rows can be noisy
                pass 

        print(f"Scraped {len(listings_data)} current {condition} listings.")

    except TimeoutException:
          print(f"Timeout: Current {condition} Listings table not found or visible at XPath {table_xpath}. Skipping listing data.")
    except Exception as e:
        print(f"Error during current {condition} listings extraction: {type(e).__name__}: {e}. Skipping listing data.")
          
    return listings_data

def scrape_price_guide_data(url):
    """
    Navigates to the URL and extracts Summary Stats, Sales History, and Current Listings.
    """
    driver = setup_driver()
    if not driver:
        return [], [], [], [], [], []
    
    historical_sales_data = []
    current_new_listings = []
    current_used_listings = []
    history_summary_stats = []
    new_available_summary_stats = []
    used_available_summary_stats = []

    try:
        print(f"Navigating to: {url}")
        driver.get(url)

        handle_popups(driver)
        
        # Check for immediate block
        try:
            driver.find_element(By.XPATH, BOT_BLOCK_MESSAGE_XPATH)
            print("!!! FATAL ERROR: BOT BLOCK DETECTED !!!")
            return [], [], [], [], [], []
        except NoSuchElementException:
            pass 

        # 1. Click the Price Guide tab 
        MAX_CLICK_RETRIES = 2
        click_success = False
        
        for attempt in range(MAX_CLICK_RETRIES):
            try:
                if attempt > 0:
                    print(f"Price Guide tab click failed on attempt {attempt}. Retrying...")
                    driver.get(url)
                    handle_popups(driver)
                
                print("Looking for the 'Price Guide' tab element...")
                price_guide_tab = WebDriverWait(driver, 15).until(
                    EC.element_to_be_clickable((By.XPATH, PRICE_GUIDE_LINK_XPATH))
                )
                
                driver.execute_script("arguments[0].click();", price_guide_tab)
                print("Successfully executed JavaScript click on the 'Price Guide' tab.")
                
                print(f"Waiting for main Price Guide container: {SALES_CONTAINER_XPATH}...")
                WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.XPATH, SALES_CONTAINER_XPATH))
                )
                print("Main Price Guide content container loaded.")
                click_success = True
                break

            except Exception as e:
                print(f"Click attempt {attempt+1} failed: {type(e).__name__}. Preparing to retry...")
                if attempt == MAX_CLICK_RETRIES - 1:
                    print("Exceeded max click retries. Proceeding, but the page may not be fully loaded.")
        
        if not click_success:
              print("Failed to load the main Price Guide container after all attempts.")
              return [], [], [], [], [], []

        # CRITICAL FIX: Sufficient sleep to allow the detailed AJAX call for sales history/listings to finish
        print("Waiting 8 seconds for detailed sales history/listings AJAX calls to complete...") 
        time.sleep(8)
        
        driver.execute_script("window.scrollBy(0, 50);")
        time.sleep(1) 

        
        # --- 2. Extract All 4 Summary Statistics ---
        (history_summary_stats, 
         new_available_summary_stats, 
         used_available_summary_stats) = scrape_summary_data(driver)
        
        
        # --- 3. Extract NEW and USED Sales History (Monthly Breakdown) ---
        
        print(f"\nAttempting to scrape USED Sales History...")
        try:
            used_table = driver.find_element(By.XPATH, USED_SALES_TABLE_XPATH)
            used_records = extract_monthly_sales_data(used_table, "USED")
            historical_sales_data.extend(used_records)
            print(f"Successfully extracted {len(used_records)} USED records from the detailed table.")

        except NoSuchElementException:
            print("No detailed USED sales history table found. Skipping USED data.")
        except Exception as e:
            print(f"Error during USED data extraction: {type(e).__name__}: {e}. Skipping USED data.")
            

        print(f"\nAttempting to scrape NEW Sales History...")
        try:
            new_table = driver.find_element(By.XPATH, NEW_SALES_TABLE_XPATH)
            new_records = extract_monthly_sales_data(new_table, "NEW")
            historical_sales_data.extend(new_records)
            print(f"Successfully extracted {len(new_records)} NEW records from the detailed table.")

        except NoSuchElementException:
            print("No detailed NEW sales history table found. Skipping NEW data.")
        except Exception as e:
            print(f"Error during NEW data extraction: {type(e).__name__}: {e}. Skipping NEW data.")
        
        print(f"Scraped total of {len(historical_sales_data)} historical sales records.")


        # --- 4. Extract CURRENT Listings ---
        
        # The timeout for finding the table is now 20 seconds inside this function
        current_new_listings = extract_current_listings(driver, "NEW", NEW_LISTINGS_TABLE_XPATH)
        current_used_listings = extract_current_listings(driver, "USED", USED_LISTINGS_TABLE_XPATH)


    except TimeoutException:
        print(f"Error: Timed out waiting for page elements at {url}. Ensure the Price Guide tab loads.")
    except Exception as e:
        print(f"An unexpected error occurred: {type(e).__name__}: {e}")
    finally:
        if driver:
            print("Closing browser window...")
            driver.quit()
        
    return (historical_sales_data, current_new_listings, current_used_listings, 
            history_summary_stats, new_available_summary_stats, used_available_summary_stats)

def save_to_csv(data, filename, fieldnames):
    """Writes the extracted data to a CSV file."""
    if not data:
        print(f"No data to save for {filename}.")
        return

    try:
        # Determine the correct fieldnames, especially for the summary files
        final_fieldnames = fieldnames
        if "stats" in filename and data:
            all_keys = set(fieldnames)
            for record in data:
                all_keys.update(record.keys())
            
            standard_stat_fields = ['Condition_Type', 'Total_Lots', 'Total_Qty', 'Min_Price', 'Avg_Price', 'Qty_Avg_Price', 'Max_Price', 'Times_Sold']
            final_fieldnames = sorted(list(all_keys), key=lambda x: standard_stat_fields.index(x) if x in standard_stat_fields else len(standard_stat_fields))

        with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=final_fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(data)
        print(f"Successfully saved {len(data)} records to {filename}")
    except IOError as e:
        print(f"I/O Error saving file {filename}: {e}")

if __name__ == '__main__':
    
    historical_fields = ['Condition', 'Month', 'Quantity', 'Price_per_unit', 'Raw_Price_Text']
    listings_fields = ['Condition', 'Store_Name', 'Quantity', 'Price_per_unit', 'Raw_Price_Text', 'Store_URL']
    # Common fields for all summary stats
    summary_fields = ['Condition_Type', 'Times_Sold', 'Total_Qty', 'Min_Price', 'Avg_Price', 'Qty_Avg_Price', 'Max_Price', 'Total_Lots']
    
    print(f"--- Starting Scraper for Item: {ITEM_NAME} ({ITEM_ID}/{COLOR_ID}) ---")
    
    # 2. Scrape the data
    (sales_history, current_new_listings, current_used_listings, 
     history_summary_stats, new_available_summary_stats, 
     used_available_summary_stats) = scrape_price_guide_data(TARGET_URL)
    
    # 3. Save the data to CSV
    
    # A. Historical Sales (Combined file)
    save_to_csv(sales_history, OUTPUT_HISTORY_FILE, historical_fields)
    
    # B. History Summary Stats (Columns 1 & 2)
    save_to_csv(history_summary_stats, OUTPUT_SUMMARY_FILE, summary_fields)
    
    # C. NEW Available Stats (New File)
    save_to_csv(new_available_summary_stats, OUTPUT_NEW_STATS_FILE, summary_fields)
    
    # D. USED Available Stats (New File)
    save_to_csv(used_available_summary_stats, OUTPUT_USED_STATS_FILE, summary_fields)
    
    # E. Current Listings (Combined into the existing file names for simplicity, but split)
    save_to_csv(current_new_listings, f'{ITEM_ID}_current_new_listings.csv', listings_fields)
    save_to_csv(current_used_listings, OUTPUT_LISTINGS_FILE, listings_fields)


    print("\n--- Scraper Execution Complete ---")

--- Starting Scraper for Item: Window 1 x 2 x 2 with Fixed Trans-Clear Glass (7026c01/11) ---
Navigating to: https://www.bricklink.com/v2/catalog/catalogitem.page?P=7026c01&idColor=11#T=P&C=11
Checking for Age Verification pop-up and preparing to inject birth year '1983'...
Found 4 potential digit fields. Injecting digits '1983'...
Error handling DOB modal: NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=143.0.7499.109)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff6661d8245
	0x7ff6661d82a0
	0x7ff665fb165d
	0x7ff665f891b1
	0x7ff6660397e6
	0x7ff66605a5c2
	0x7ff665ffac29
	0x7ff665ffba93
	0x7ff6664effe0
	0x7ff6664ea920
	0x7ff666509086
	0x7ff6661f5744
	0x7ff6661fe6ec
	0x7ff6661e1964
	0x7ff6661e1b15
	0x7ff6661c7842
	0x7ffd7aade8d7
	0x7ffd7c42c53c
. Continuing...
Checking for Cookie consent banner...
Error handling cookie banner: NoSuchWindowException. Continuing...
Pop-up han

In [4]:
!pip show selenium

Name: selenium
Version: 4.37.0
Summary: Official Python bindings for Selenium WebDriver
Home-page: https://www.selenium.dev
Author: 
Author-email: 
License: Apache-2.0
Location: C:\Users\sezar\miniforge3\Lib\site-packages
Requires: certifi, trio, trio-websocket, typing_extensions, urllib3, websocket-client
Required-by: 


In [49]:
import time
import re
import csv
import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service 
from selenium.common.exceptions import TimeoutException, WebDriverException, NoSuchElementException, ElementClickInterceptedException
import pandas as pd
import subprocess
import os 

# --- CRITICAL CONFIGURATION: UPDATE THESE PATHS ---
# 1. CHROME_BINARY_PATH: MUST point to the specific 'chrome.exe' executable (the browser).
# 2. CHROME_DRIVER_PATH: MUST point to the specific 'chromedriver.exe' executable (the driver).
CHROME_BINARY_PATH = r"C:\Program Files\Google\Chrome\Application\chrome.exe" 
CHROME_DRIVER_PATH = r"C:\Pricing_BL\chromedriver.exe" # <-- CRITICAL: Ensure this is chromedriver.exe

# --- Scraper Debugging Flag ---
HEADLESS_MODE = False 

# --- Item Configuration (Used by scrape_stats in __main__) ---
# NOTE: To test the de-duplication logic, run the script once. The second run will skip 
# summary stats but perform the listings scrape.
ITEM_ID = "x467c12" 
ITEM_ID = "7026c01"
COLOR_ID = "9"
ITEM_NAME = "Window 1 x 2 x 2 with Fixed Trans-Clear Glass (Example Name)" 

TIMEOUT = 20 # seconds

# --- CSS/XPATH Selectors ---
PRICE_GUIDE_LINK_XPATH = "//td[text()='Price Guide']" 
SALES_CONTAINER_XPATH = "//div[@id='_idPGContents']" 
BOT_BLOCK_MESSAGE_XPATH = "//div[contains(@class, 'blp-block-message') or contains(text(), 'Access Denied')]"

# Summary Statistics (table[1]) - All 4 columns in the first main row (tr[1])
NEW_HISTORY_SUMMARY_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[1]/table[1]"
USED_HISTORY_SUMMARY_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[2]/table[1]"
NEW_AVAILABLE_SUMMARY_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[3]/table[1]"
USED_AVAILABLE_SUMMARY_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[4]/table[1]" 

# Sales History (pcipgInnerTable in td[1] or td[2]) - Second main row (tr[2])
NEW_SALES_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[1]//table[contains(@class, 'pcipgInnerTable')]"
USED_SALES_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[2]//table[contains(@class, 'pcipgInnerTable')]"

# Current Listings (pcipgInnerTable in td[3] or td[4]) - Second main row (tr[2])
NEW_LISTINGS_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[3]//table[contains(@class, 'pcipgInnerTable')]"
USED_LISTINGS_TABLE_XPATH = f"{SALES_CONTAINER_XPATH}//td[4]//table[contains(@class, 'pcipgInnerTable')]" 

# Current Listing Rows: Tr that contains a link to a store in the first column
USED_LISTING_ROW_XPATH = ".//tr[td[1]/a[contains(@href, 'store.asp?sID')]]"


DOB_INPUT_ELEMENTS_XPATH = "//input[contains(@class, 'blp-age-gate__input-field')]"
DOB_SUBMIT_XPATH = "//button[normalize-space(.)='Submit' or normalize-space(.)='Enter' or normalize-space(.)='Continue']"
COOKIES_ACCEPT_XPATH = "//button[contains(text(), 'Accept all')]"

# --- Global Field Definitions ---
STATS_FIELDS = [
    'item_id', 
    'color_id', 
    'Quantity sold in used condition', 
    'Quantity sold in new condition',
    'Quantity available in used condition',
    'Quantity available in new condition'
]


def handle_popups(driver):
    """Handles the age verification and cookie consent modals."""
    
    # 1. Handle Age Verification
    try:
        time.sleep(1.5) 
        print("Checking for Age Verification pop-up and preparing to inject birth year '1983'...")
        
        digit_inputs = WebDriverWait(driver, 7).until( 
            EC.presence_of_all_elements_located((By.XPATH, DOB_INPUT_ELEMENTS_XPATH))
        )
        
        num_fields = len(digit_inputs)

        if num_fields > 0:
            year_str = "1983"
            digits_to_use = year_str[-num_fields:] 
            
            print(f"Found {len(digit_inputs)} potential digit fields. Injecting digits '{digits_to_use}'...")
            
            js_script = """
            arguments[0].value = arguments[1]; 
            arguments[0].dispatchEvent(new Event('focus', { bubbles: true }));
            arguments[0].dispatchEvent(new Event('input', { bubbles: true })); 
            arguments[0].dispatchEvent(new Event('change', { bubbles: true }));
            arguments[0].dispatchEvent(new Event('blur', { bubbles: true }));
            """
            
            for i in range(num_fields):
                digit = digits_to_use[i]
                input_field = digit_inputs[i]
                driver.execute_script(js_script, input_field, digit)
                time.sleep(0.1) 
            
            dob_submit = WebDriverWait(driver, 5).until( 
                EC.element_to_be_clickable((By.XPATH, DOB_SUBMIT_XPATH))
            )
            driver.execute_script("arguments[0].click();", dob_submit)
            print("Submitted birth year.")
            time.sleep(3) 
        
        else:
             print(f"Did not find any potential digit inputs. Skipping digit input.")
        
    except TimeoutException:
        print("Age verification modal not found or not needed.")
    except Exception as e:
        print(f"Error handling DOB modal: {type(e).__name__}: {e}. Continuing...")


    # 2. Handle Cookie Consent
    try:
        print("Checking for Cookie consent banner...")
        cookie_accept_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, COOKIES_ACCEPT_XPATH))
        )
        driver.execute_script("arguments[0].click();", cookie_accept_button)
        print("Accepted all cookies.")
        time.sleep(1) 
    except TimeoutException:
        print("No cookie consent banner detected.")
    except Exception as e:
        print(f"Error handling cookie banner: {type(e).__name__}. Continuing...")

    print("Pop-up handling complete.")


def setup_driver():
    """Initializes and returns a Selenium WebDriver instance."""
    try:
        chrome_options = webdriver.ChromeOptions()
        
        # --- Human-like Options for Bot Detection Bypass ---
        chrome_options.add_argument("window-size=1280,800")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        # Ignores the 'Chrome is being controlled by automated software' message
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_experimental_option('useAutomationExtension', False)
        chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
        
        # Toggle headless mode based on the flag
        if HEADLESS_MODE:
            chrome_options.add_argument("--headless")
        
        if CHROME_DRIVER_PATH and CHROME_BINARY_PATH:
            chrome_options.binary_location = CHROME_BINARY_PATH
            service = Service(CHROME_DRIVER_PATH)
            driver = webdriver.Chrome(service=service, options=chrome_options)
        else:
            driver = webdriver.Chrome(options=chrome_options)
            
        driver.set_page_load_timeout(TIMEOUT)
        return driver
    except WebDriverException as e:
        print(f"Error initializing WebDriver: {e}")
        return None

def extract_price(text):
    """Cleans and extracts the numeric price value from a text string, handling various currencies."""
    if not text or '-' in text:
        return None
    
    # Remove currency symbols/text, including USD, CA, etc.
    cleaned_text = re.sub(r'(\$|€|£|AUD|CAD|US|CA)\s*', '', text, flags=re.IGNORECASE)
    cleaned_text = cleaned_text.replace('~', '').strip()
    
    try:
        match = re.search(r'\d+\.?\d*', cleaned_text)
        if match:
             return float(match.group(0))
        return None
    except ValueError:
        return None 

def extract_monthly_sales_data(table_element, condition):
    """
    HELPER: Extracts the detailed Qty and Price/Unit lines from the nested monthly sales table.
    """
    sales_rows = table_element.find_elements(By.TAG_NAME, "tr")
    extracted_records = []
    current_month = "N/A"
    
    # Keywords that indicate a summary line or header line, not a detailed sale line
    summary_keywords = {'Times Sold:', 'Total Lots:', 'Total Qty:', 'Min Price:', 'Avg Price:', 'Qty Avg Price:', 'Max Price:', 'Qty', 'Each'}

    for row in sales_rows:
        cells = row.find_elements(By.TAG_NAME, "td")
        
        row_text = [cell.text.strip() for cell in cells]
        
        if not any(row_text):
            continue

        # Rule 1: Identify the Month/Year Header (e.g., "December 2025")
        if len(cells) > 0 and 'pcipgSubHeader' in cells[0].get_attribute('class'):
            current_month = cells[0].text.strip()
            continue
        
        # Rule 2: Identify detailed sales rows (Qty and Price)
        if len(cells) == 3:
            qty_text = cells[1].text.strip() 
            price_text = cells[2].text.strip() 

            # Check if it's a valid sales record row (not a header like 'Qty' 'Each')
            if qty_text and price_text and qty_text.isdigit():
                
                price = extract_price(price_text)
                
                extracted_records.append({
                    'Condition': condition,
                    'Month': current_month,
                    'Quantity': qty_text,
                    'Price_per_unit': price,
                    'Raw_Price_Text': price_text
                })
        
        # Rule 3: Skip the summary rows that appear after each month
        elif len(cells) == 2:
            cell_0_text = cells[0].text.strip() 
            if any(cell_0_text.startswith(kw) for kw in summary_keywords):
                continue

    return extracted_records


def scrape_summary_data(driver):
    """
    HELPER: Extracts the summary statistics from all four columns in the first row.
    Returns: (history_stats, new_available_stats, used_available_stats)
    """
    all_stats = []
    
    # Define all 4 columns to scrape in the first row
    summary_configs = [
        ("NEW_HISTORY", NEW_HISTORY_SUMMARY_TABLE_XPATH), 
        ("USED_HISTORY", USED_HISTORY_SUMMARY_TABLE_XPATH),
        ("NEW_AVAILABLE", NEW_AVAILABLE_SUMMARY_TABLE_XPATH),
        ("USED_AVAILABLE", USED_AVAILABLE_SUMMARY_TABLE_XPATH)
    ]
    
    for stat_type, xpath in summary_configs:
        print(f"\nAttempting to scrape {stat_type} Summary Data...")
        
        try:
            summary_table = WebDriverWait(driver, 5).until(
                EC.visibility_of_element_located((By.XPATH, xpath))
            )
            
            rows = summary_table.find_elements(By.TAG_NAME, "tr")
            record = {'Condition_Type': stat_type}
            
            for row in rows:
                cells = row.find_elements(By.TAG_NAME, "td")
                if len(cells) == 2:
                    key = cells[0].text.strip().replace(':', '')
                    value = cells[1].text.strip()
                    field_name = key.replace(' ', '_')
                    record[field_name] = value
            
            if len(record) > 1:
                all_stats.append(record)
                print(f"Successfully extracted {len(record)-1} fields for {stat_type} summary.")
            
        except TimeoutException:
            print(f"Timeout: {stat_type} Summary Table was not found at XPath: {xpath}. Skipping.")
        except Exception as e:
            print(f"Error during {stat_type} summary extraction: {type(e).__name__}: {e}. Skipping.")

    # Separate the results for return values
    history_stats = [s for s in all_stats if 'HISTORY' in s['Condition_Type']]
    new_available_stats = [s for s in all_stats if s['Condition_Type'] == 'NEW_AVAILABLE']
    used_available_stats = [s for s in all_stats if s['Condition_Type'] == 'USED_AVAILABLE']
    
    return history_stats, new_available_stats, used_available_stats


def extract_current_listings(driver, condition, table_xpath):
    """
    HELPER: Extracts current listings from the specified table XPath.
    """
    listings_data = []
    print(f"\nScraping Current {condition} Listings...")
    
    try:
        current_listings_table = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, table_xpath))
        )
        
        # 2. Force scroll the table into view using JavaScript to resolve visibility
        driver.execute_script("arguments[0].scrollIntoView(true);", current_listings_table)
        time.sleep(1) # Small pause after scrolling

        # 3. Find listing rows inside the specific inner table
        listing_rows = current_listings_table.find_elements(By.XPATH, USED_LISTING_ROW_XPATH)
        
        first_row_printed = False 
        
        for row in listing_rows:
            try:
                if not first_row_printed:
                    print(f"HTML of first successful {condition} listing row:\n{row.get_attribute('outerHTML')}")
                    first_row_printed = True
                
                cols = row.find_elements(By.TAG_NAME, 'td')
                
                if len(cols) >= 3:
                    
                    store_link = cols[0].find_element(By.TAG_NAME, 'a')
                    
                    try:
                       store_name = store_link.find_element(By.TAG_NAME, 'img').get_attribute('title').replace('Store: ', '').strip()
                    except NoSuchElementException:
                       store_name = store_link.text.strip() 
                       
                    store_url = store_link.get_attribute('href')
                    
                    qty = cols[1].text.strip()
                    raw_price = cols[2].text.strip()
                    price = extract_price(raw_price)
                    
                    listings_data.append({
                        'Condition': condition,
                        'Store_Name': store_name,
                        'Quantity': qty,
                        'Price_per_unit': price,
                        'Raw_Price_Text': raw_price,
                        'Store_URL': store_url
                    })
                    
            except Exception:
                pass 

        print(f"Scraped {len(listings_data)} current {condition} listings.")

    except TimeoutException:
         print(f"Timeout: Current {condition} Listings table not found or visible at XPath {table_xpath}. Skipping listing data.")
    except Exception as e:
         print(f"Error during current {condition} listings extraction: {type(e).__name__}: {e}. Skipping listing data.")
         
    return listings_data


def _setup_and_navigate(item_id, color_id):
    """
    INTERNAL HELPER: Sets up the driver, navigates to the URL, and clicks the Price Guide tab.
    Returns the active driver instance or None on failure.
    """
    url = f"https://www.bricklink.com/v2/catalog/catalogitem.page?P={item_id}&idColor={color_id}#T=P&C={color_id}"
    driver = setup_driver()
    if not driver:
        return None

    try:
        print(f"Navigating to: {url}")
        driver.get(url)
        handle_popups(driver)

        # Check for immediate block
        try:
            driver.find_element(By.XPATH, BOT_BLOCK_MESSAGE_XPATH)
            print("!!! FATAL ERROR: BOT BLOCK DETECTED !!!")
            driver.quit()
            return None
        except NoSuchElementException:
            pass

        # 1. Click the Price Guide tab 
        MAX_CLICK_RETRIES = 2
        click_success = False
        
        for attempt in range(MAX_CLICK_RETRIES):
            try:
                if attempt > 0:
                    print(f"Price Guide tab click failed on attempt {attempt}. Retrying...")
                    driver.get(url)
                    handle_popups(driver)
                
                print("Looking for the 'Price Guide' tab element...")
                price_guide_tab = WebDriverWait(driver, 15).until(
                    EC.element_to_be_clickable((By.XPATH, PRICE_GUIDE_LINK_XPATH))
                )
                
                driver.execute_script("arguments[0].click();", price_guide_tab)
                print("Successfully executed JavaScript click on the 'Price Guide' tab.")
                
                print(f"Waiting for main Price Guide container: {SALES_CONTAINER_XPATH}...")
                WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.XPATH, SALES_CONTAINER_XPATH))
                )
                print("Main Price Guide content container loaded.")
                click_success = True
                break

            except Exception as e:
                print(f"Click attempt {attempt+1} failed: {type(e).__name__}. Preparing to retry...")
                if attempt == MAX_CLICK_RETRIES - 1:
                    print("Exceeded max click retries. Returning None.")
                    driver.quit()
                    return None
        
        if not click_success:
             print("Failed to load the main Price Guide container after all attempts. Returning None.")
             driver.quit()
             return None

        print("Waiting 8 seconds for detailed sales history/listings AJAX calls to complete...") 
        time.sleep(8) 
        
        driver.execute_script("window.scrollBy(0, 50);")
        time.sleep(1) 
        
        return driver

    except Exception as e:
        print(f"An unexpected error occurred during navigation: {type(e).__name__}: {e}")
        if driver:
             driver.quit()
        return None

# --- REQUIRED MODULAR METHODS ---

def scrape_used_sales_history(driver):
    """Scrapes the detailed sales history for USED items."""
    print(f"\nAttempting to scrape USED Sales History (Detailed Monthly Breakdown)...")
    try:
        used_table = driver.find_element(By.XPATH, USED_SALES_TABLE_XPATH)
        used_records = extract_monthly_sales_data(used_table, "USED")
        print(f"Successfully extracted {len(used_records)} USED sales history records.")
        return used_records
    except NoSuchElementException:
        print("No detailed USED sales history table found. Returning empty list.")
        return []
    except Exception as e:
        print(f"Error during USED sales history extraction: {type(e).__name__}: {e}. Returning empty list.")
        return []

def scrape_new_sales_history(driver):
    """Scrapes the detailed sales history for NEW items."""
    print(f"\nAttempting to scrape NEW Sales History (Detailed Monthly Breakdown)...")
    try:
        new_table = driver.find_element(By.XPATH, NEW_SALES_TABLE_XPATH)
        new_records = extract_monthly_sales_data(new_table, "NEW")
        print(f"Successfully extracted {len(new_records)} NEW sales history records.")
        return new_records
    except NoSuchElementException:
        print("No detailed NEW sales history table found. Returning empty list.")
        return []
    except Exception as e:
        print(f"Error during NEW sales history extraction: {type(e).__name__}: {e}. Returning empty list.")
        return []

def scrape_used_available_listings(driver):
    """Scrapes the currently available USED listings."""
    return extract_current_listings(driver, "USED", USED_LISTINGS_TABLE_XPATH)

def scrape_new_available_listings(driver):
    """Scrapes the currently available NEW listings."""
    return extract_current_listings(driver, "NEW", NEW_LISTINGS_TABLE_XPATH)

# --- END OF MODULAR METHODS ---

def get_base_filename():
    """Generates the YYYY_MM prefix for the current month's files."""
    now = datetime.datetime.now()
    year = now.year
    month = f"{now.month:02d}" 
    return f"{year}_{month}"

def generate_listings_filename(condition, item_id, color_id):
    """Generates the required CSV filename for detailed listings: YYYY_MM_[new/used]_listings_ITEMID_COLORID.csv"""
    base = get_base_filename()
    return f"{base}_{condition.lower()}_listings_{item_id}_{color_id}.csv"

def scrape_stats(item_id, color_id):
    """
    Handles de-duplication, orchestrates scraping of listings (always runs) and 
    summary stats (runs only if data is missing from the master CSV).
    """
    
    # 1. Determine consolidated stats filename
    stats_filename = f"{get_base_filename()}_stats.csv"
    
    # Initialize variables for return
    historical_sales_data = []
    current_new_listings = []
    current_used_listings = []
    history_summary_stats = []
    new_available_summary_stats = []
    used_available_summary_stats = []
    
    # 2. Check for existing data (De-duplication Logic)
    try:
        # Read existing file or create empty DataFrame
        if os.path.exists(stats_filename):
            df = pd.read_csv(stats_filename, dtype={'item_id': str, 'color_id': str})
        else:
            df = pd.DataFrame(columns=STATS_FIELDS)
            
        # Check if item_id and color_id combination already exists
        stats_exists = ((df['item_id'] == item_id) & (df['color_id'] == color_id)).any()
            
    except Exception as e:
        print(f"Error reading/checking stats file {stats_filename}: {e}. Treating as 'stats not found'.")
        stats_exists = False
        df = pd.DataFrame(columns=STATS_FIELDS)

    # 3. Setup and Navigate (ALWAYS runs to get current listings)
    print(f"\n--- Starting Scrape for {item_id}/{color_id} ---")
    driver = _setup_and_navigate(item_id, color_id)
    if not driver:
        # If navigation fails, we return empty sets for everything.
        print("Navigation failed. Aborting scrape.")
        return historical_sales_data, current_new_listings, current_used_listings, history_summary_stats, new_available_summary_stats, used_available_summary_stats
    
    # --- Logic Branch ---
    if not stats_exists:
        # --- PATH A: Full Scrape (Stats, History, and Listings) ---
        print(f"Stats not found in {stats_filename}. Performing full scrape.")
        
        # 4. Scrape Summary Data (History and Availability)
        (history_summary_stats, 
         new_available_summary_stats, 
         used_available_summary_stats) = scrape_summary_data(driver)
        
        # 5. Extract and Prepare New Stats Row
        used_sales_qty = next((s.get('Total_Qty') for s in history_summary_stats if s.get('Condition_Type') == 'USED_HISTORY'), 'N/A')
        new_sales_qty = next((s.get('Total_Qty') for s in history_summary_stats if s.get('Condition_Type') == 'NEW_HISTORY'), 'N/A')
        used_available_qty = next((s.get('Total_Qty') for s in used_available_summary_stats if s.get('Condition_Type') == 'USED_AVAILABLE'), 'N/A')
        new_available_qty = next((s.get('Total_Qty') for s in new_available_summary_stats if s.get('Condition_Type') == 'NEW_AVAILABLE'), 'N/A')

        new_row_data = {
            'item_id': item_id,
            'color_id': color_id,
            'Quantity sold in used condition': used_sales_qty,
            'Quantity sold in new condition': new_sales_qty,
            'Quantity available in used condition': used_available_qty,
            'Quantity available in new condition': new_available_qty
        }
        
        # 6. Append and Save Consolidated Stats CSV
        new_df = pd.DataFrame([new_row_data])
        df = pd.concat([df, new_df], ignore_index=True)
        df.to_csv(stats_filename, index=False)
        print(f"Successfully saved stats for {item_id}/{color_id} to consolidated file: {stats_filename}")
        
        # 7. Scrape Detailed Sales History (Only needed when history/stats are new)
        used_sales_records = scrape_used_sales_history(driver)
        new_sales_records = scrape_new_sales_history(driver)
        historical_sales_data.extend(used_sales_records)
        historical_sales_data.extend(new_sales_records)
        
    else:
        # --- PATH B: Listings Only Scrape ---
        print(f"Stats for {item_id}/{color_id} already exist. Skipping summary stats and history scrape.")
    
    # 7b. Scrape Detailed Available Listings (ALWAYS run)
    current_new_listings = scrape_new_available_listings(driver)
    current_used_listings = scrape_used_available_listings(driver)
    
    print(f"\nTotal scraped listings: New={len(current_new_listings)}, Used={len(current_used_listings)}.")
    
    # 8. Quit Driver
    print("Closing browser window...")
    driver.quit()
    
    # Return all data (skipped data will be empty lists/dicts)
    return (historical_sales_data, current_new_listings, current_used_listings, 
            history_summary_stats, new_available_summary_stats, used_available_summary_stats)


def save_to_csv(data, filename, fieldnames):
    """Writes the extracted data to a CSV file."""
    if not data:
        print(f"No data to save for {filename}.")
        return

    try:
        with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(data)
        print(f"Successfully saved {len(data)} records to {filename}")
    except IOError as e:
        print(f"I/O Error saving file {filename}: {e}")

if __name__ == '__main__':
    
    listings_fields = ['Condition', 'Store_Name', 'Quantity', 'Price_per_unit', 'Raw_Price_Text', 'Store_URL']
    
    print(f"--- Starting Primary Execution for Item: {ITEM_NAME} ({ITEM_ID}/{COLOR_ID}) ---")
    
    # --- Primary Call: Call the single orchestrator method ---
    (sales_history, current_new_listings, current_used_listings, 
     history_summary_stats, new_available_summary_stats, 
     used_available_summary_stats) = scrape_stats(ITEM_ID, COLOR_ID)
    
    # --- Post-Processing (Saving the detailed listing data using the new format) ---
    if current_new_listings:
        new_listings_filename = generate_listings_filename('new', ITEM_ID, COLOR_ID)
        save_to_csv(current_new_listings, new_listings_filename, listings_fields)
    
    if current_used_listings:
        used_listings_filename = generate_listings_filename('used', ITEM_ID, COLOR_ID)
        save_to_csv(current_used_listings, used_listings_filename, listings_fields)


    print("\n--- Scraper Execution Complete ---")

--- Starting Primary Execution for Item: Window 1 x 2 x 2 with Fixed Trans-Clear Glass (Example Name) (7026c01/9) ---

--- Starting Scrape for 7026c01/9 ---
Navigating to: https://www.bricklink.com/v2/catalog/catalogitem.page?P=7026c01&idColor=9#T=P&C=9
Checking for Age Verification pop-up and preparing to inject birth year '1983'...
Found 4 potential digit fields. Injecting digits '1983'...
Age verification modal not found or not needed.
Checking for Cookie consent banner...
Accepted all cookies.
Pop-up handling complete.
Looking for the 'Price Guide' tab element...
Successfully executed JavaScript click on the 'Price Guide' tab.
Waiting for main Price Guide container: //div[@id='_idPGContents']...
Main Price Guide content container loaded.
Waiting 8 seconds for detailed sales history/listings AJAX calls to complete...
Stats not found in 2025_12_stats.csv. Performing full scrape.

Attempting to scrape NEW_HISTORY Summary Data...
Successfully extracted 6 fields for NEW_HISTORY summary.

In [ ]:
Keep my code as above and do minor changes, as little as possible! Ok??

I want to add:


2) write four different methods to scrape the data for available used

available new
sales history used
sales history new

variables are item_id, color_id


3) for example, call all methods in the main

My important question:

Can you write this in a way they don't do any duplicate scraping work?

In [44]:

df = pd.read_csv(r"C:\Pricing_BL\2025_12_used_listings_x467c12_11.csv")
# df[df["Condition"]=="NEW"]
# df.iloc[20:40]
df

,Condition,Store_Name,Quantity,Price_per_unit,Raw_Price_Text,Store_URL
0,USED,CSimian's Brick Emporium,1,6.90,~CA $6.90,http://www.bricklink.com/store.asp?sID=151997&...
1,USED,3BBricks Saigon,1,6.90,~CA $6.90,http://www.bricklink.com/store.asp?sID=3615860...
2,USED,Battle Platform,1,7.66,~CA $7.66,http://www.bricklink.com/store.asp?sID=23918&i...
3,USED,Brickotopia,1,8.28,~CA $8.28,http://www.bricklink.com/store.asp?sID=166715&...
4,USED,StarBricks95,14,8.50,~CA $8.50,http://www.bricklink.com/store.asp?sID=269414&...
...,...,...,...,...,...,...
89,USED,SillyJillies,2,20.70,~CA $20.70,http://www.bricklink.com/store.asp?sID=1864026...
90,USED,1mBuriedButAlive,2,20.70,~CA $20.70,http://www.bricklink.com/store.asp?sID=1442420...
91,USED,WILCO Toys,7,24.15,~CA $24.15,http://www.bricklink.com/store.asp?sID=2750172...
92,USED,Allerhand und Gwand,1,29.44,~CA $29.44,http://www.bricklink.com/store.asp?sID=674312&...


In [53]:
path = r"C:\Pricing_BL\2025_12_stats.csv"
df = pd.read_csv(path)

# df.iloc[20:40]
df

,item_id,color_id,Quantity sold in used condition,Quantity sold in new condition,Quantity available in used condition,Quantity available in new condition
0,x467c12,11,206,2,282,4
1,7026c01,11,76,0,253,81
2,7026c01,9,205,0,1017,102


In [52]:
path = r"C:\Pricing_BL\x467c12_current_new_listings.csv"
df = pd.read_csv(path)
# df.iloc[20:40]
df

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Pricing_BL\\x467c12_current_new_listings.csv'

In [ ]:
path = r"C:\Pricing_BL\statistics\25_12_stats.csv"
df = pd.read_csv(path)
# df.iloc[20:40]
df


,timestamp,item_no,color_id,new_stock_avg_price,new_stock_qty_avg_price,new_stock_unit_quantity,new_stock_total_quantity,used_stock_avg_price,used_stock_qty_avg_price,used_stock_unit_quantity,used_stock_total_quantity,new_sold_avg_price,new_sold_qty_avg_price,new_sold_unit_quantity,new_sold_total_quantity,used_sold_avg_price,used_sold_qty_avg_price,used_sold_unit_quantity,used_sold_total_quantity
0,2025-12-17 02:09:37,7026c01,11,27.7325,24.0987,4,81,12.1178,12.4668,94,243,0.0000,0.0000,0,0,8.0264,7.9590,39,77
1,2025-12-17 02:15:07,7026c01,5,1.2587,1.0697,69,3879,0.0906,0.1076,2062,49237,0.3349,0.3349,2,4,0.0586,0.0596,248,2650
2,2025-12-17 03:27:36,7026c01,7,6.1315,4.7843,20,296,0.5044,0.6178,640,2297,1.1585,1.1570,2,10,0.4063,0.3951,120,328


In [ ]:
path = r"price_guide_set/76976-1_N.csv"
df = pd.read_csv(path)
# df.iloc[20:40]
df

# {"set_no":"75387-1","condition":"N","projected_sale":15.0067,"part_out_value":35.617,"total_parts_processed":159,"message":null}

,part_no,item_type,color_id,quantity,sold_avg_price,sold_total_quantity,stock_total_quantity,probability,expected_return
0,2357,PART,2,8,0.0518,79513,401614,0.1980,0.0820
1,2357,PART,155,9,0.2440,1846,41505,0.0445,0.0977
2,2417,PART,36,1,0.1554,43795,316868,0.1382,0.0215
3,2420,PART,2,4,0.0573,78791,272920,0.2887,0.0662
4,2420,PART,85,4,0.0626,134466,554857,0.2423,0.0607
...,...,...,...,...,...,...,...,...,...
359,quetzal02,PART,55,1,23.3602,81,131,0.6136,14.3347
360,jw134,MINIFIG,0,1,6.2923,115,227,0.5044,3.1737
361,jw135,MINIFIG,0,1,7.9283,135,193,0.6959,5.5171
362,jw136,MINIFIG,0,1,7.8569,141,176,0.7966,6.2589


In [71]:
import pandas as pd
import io

path = r"C:\Pricing_BL\price_guide_set\75387-1_N.csv"
df = pd.read_csv(path)
# df.iloc[20:40]
print("--- Loaded DataFrame Head ---")
print(df.head())
print("\n" + "="*40 + "\n")

# --- SOLUTION: Calculate the sum of (quantity * sold_avg_price) ---

# 1. Perform element-wise multiplication of the two columns.
# This creates a new temporary Pandas Series where each element is the product of the row's values.
calculated_value_per_row = df['quantity'] * df['sold_avg_price']

# 2. Calculate the sum of this new Series.
total_calculated_sum = calculated_value_per_row.sum()

print(f"The sum of (quantity * sold_avg_price) is: {total_calculated_sum:.4f}")

# You can also do this in a single line:
total_calculated_sum_oneline = (df['quantity'] * df['sold_avg_price']).sum()
print(f"One-line calculation check: {total_calculated_sum_oneline:.4f}")

--- Loaded DataFrame Head ---
  part_no item_type  color_id  quantity  sold_avg_price  sold_total_quantity  \
0    2357      PART         1         2          0.0932                81813   
1    2357      PART        86         2          0.0987               100804   
2    2420      PART         1         1          0.0536               242657   
3    2431      PART         1         5          0.0704               316017   
4    2445      PART        85         1          0.1646                26736   

   stock_total_quantity  probability  expected_return  
0                136031       0.6014           0.1121  
1                260965       0.3863           0.0763  
2                626380       0.3874           0.0208  
3               1709535       0.1849           0.0651  
4                 92983       0.2875           0.0473  


The sum of (quantity * sold_avg_price) is: 82.3112
One-line calculation check: 82.3112


In [72]:
import pandas as pd
import numpy as np

# --- Configuration for your actual environment ---
# Make sure this path is correct for your environment.
PATH = r"C:\Pricing_BL\price_guide_set\75387-1_N.csv" 
API_VALUE = 70.5173
LOCAL_VALUE = 82.3112
CALC_COLUMN = 'sold_avg_price'

try:
    # Load your actual data
    df = pd.read_csv(PATH)
    print(f"Successfully loaded DataFrame with {len(df)} rows.")
except FileNotFoundError:
    # --- FALLBACK: Simulation for demonstration (Replace with your actual data) ---
    print("Warning: CSV not found at the specified path. Using simulated data for demonstration.")
    data = {
        'quantity': [1, 2, 4, 1, 5, 10],
        'sold_avg_price': [0.0195, 0.0800, 0.0250, 3.5000, 0.0000, 0.5000],
        'item_type': ['PART', 'PART', 'PART', 'MINIFIG', 'PART', 'PART'],
        'expected_return': [0.0195 * 0.5, 0.0800 * 0.9, 0.0250 * 0.8, 3.5000 * 0.1, 0.0000, 0.5000 * 0.95],
    }
    df = pd.DataFrame(data)
    # The simulation below is set to roughly match your expected local value:
    # df['quantity'] * df['sold_avg_price'] approx 82.3112
    # df['expected_return'].sum() approx 70.5173
    
    # NOTE: Since I cannot replicate your exact numbers, the focus is on the METHODOLOGY.

print("\n" + "="*60)
print(f"--- API/Local Value Discrepancy Analysis ---")
print(f"Local Calculation (your current result): {LOCAL_VALUE:.4f}")
print(f"API Reported Value: {API_VALUE:.4f}")
print("="*60 + "\n")


# --- 1. Check User's Local Calculation ---
print("--- Check 1: Simple Qty * AvgPrice (Your current calculation) ---")
df['local_value_per_row'] = df['quantity'] * df[CALC_COLUMN]
local_total = df['local_value_per_row'].sum()
print(f"Sum of (quantity * {CALC_COLUMN}): {local_total:.4f}")

if abs(local_total - LOCAL_VALUE) < 0.0001:
    print("SUCCESS: Local calculation matches your reported 82.3112.")
else:
    print(f"WARNING: Local calculation {local_total:.4f} does NOT match your reported 82.3112. Please re-run the previous calculation to confirm the starting point.")


# --- 2. Check API Hypothesis 1: Use 'expected_return' column (Risk-Adjusted Value) ---
print("\n--- Check 2: Sum of 'expected_return' (API Hypothesis) ---")
if 'expected_return' in df.columns:
    api_hypothesis_1_total = df['expected_return'].sum()
    print(f"Sum of 'expected_return': {api_hypothesis_1_total:.4f}")
    if abs(api_hypothesis_1_total - API_VALUE) < 0.0001:
        print("\n!!! MATCH FOUND !!! -> API is likely using the 'expected_return' column (risk-adjusted value).")
    else:
        print("NO MATCH: Sum of 'expected_return' does not match the API value.")
else:
    print("Column 'expected_return' not found. Cannot check Hypothesis 1.")


# --- 3. Check API Hypothesis 2: Filtering (Exclude zero/missing prices) ---
print(f"\n--- Check 3: Filtered Sum of (quantity * {CALC_COLUMN}) ---")

# Filter out rows where sold_avg_price is zero, null, or negative
df_filtered = df[df[CALC_COLUMN] > 0]
filtered_total = (df_filtered['quantity'] * df_filtered[CALC_COLUMN]).sum()

print(f"Rows remaining after filter (Price > 0): {len(df_filtered)} / {len(df)}")
print(f"Filtered sum (Qty * AvgPrice): {filtered_total:.4f}")

if abs(filtered_total - API_VALUE) < 0.0001:
    print("\n!!! MATCH FOUND !!! -> API is likely excluding items with a sold_avg_price of 0 or less.")
else:
    print("NO MATCH: Filtered calculation does not match the API value.")

# --- 4. Check API Hypothesis 3: Filtering by Item Type (Exclude Minifigs/Extras) ---
print(f"\n--- Check 4: Filtered Sum (Only 'PART' item_type) ---")

if 'item_type' in df.columns:
    df_part_filtered = df[df['item_type'] == 'PART']
    part_filtered_total = (df_part_filtered['quantity'] * df_part_filtered[CALC_COLUMN]).sum()
    
    print(f"Rows remaining after filter (item_type == 'PART'): {len(df_part_filtered)} / {len(df)}")
    print(f"Part-only sum (Qty * AvgPrice): {part_filtered_total:.4f}")

    if abs(part_filtered_total - API_VALUE) < 0.0001:
        print("\n!!! MATCH FOUND !!! -> API is likely excluding Minifigs/Instructions/Other types.")
    else:
        print("NO MATCH: Part-only calculation does not match the API value.")
else:
    print("Column 'item_type' not found. Cannot check Hypothesis 3.")

Successfully loaded DataFrame with 159 rows.

--- API/Local Value Discrepancy Analysis ---
Local Calculation (your current result): 82.3112
API Reported Value: 70.5173

--- Check 1: Simple Qty * AvgPrice (Your current calculation) ---
Sum of (quantity * sold_avg_price): 82.3112
SUCCESS: Local calculation matches your reported 82.3112.

--- Check 2: Sum of 'expected_return' (API Hypothesis) ---
Sum of 'expected_return': 50.7335
NO MATCH: Sum of 'expected_return' does not match the API value.

--- Check 3: Filtered Sum of (quantity * sold_avg_price) ---
Rows remaining after filter (Price > 0): 159 / 159
Filtered sum (Qty * AvgPrice): 82.3112
NO MATCH: Filtered calculation does not match the API value.

--- Check 4: Filtered Sum (Only 'PART' item_type) ---
Rows remaining after filter (item_type == 'PART'): 153 / 159
Part-only sum (Qty * AvgPrice): 44.3852
NO MATCH: Part-only calculation does not match the API value.


In [4]:
"C:\Windows\SeleniumDrivers\chrome-win64\chrome.exe"

!google-chrome --version

<>:1: SyntaxWarning: invalid escape sequence '\W'
<>:1: SyntaxWarning: invalid escape sequence '\W'
C:\Users\sezar\AppData\Local\Temp\ipykernel_7120\1819534532.py:1: SyntaxWarning: invalid escape sequence '\W'
  "C:\Windows\SeleniumDrivers\chrome-win64\chrome.exe"
'google-chrome' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import sys
!chromedriver --version



'chromedriver' is not recognized as an internal or external command,
operable program or batch file.


In [7]:
!pip install selenium

You should consider upgrading via the 'c:\users\sezar\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


In [1]:
!pip install --upgrade ipykernel

  Attempting uninstall: ipykernel
    Found existing installation: ipykernel 6.30.0
    Uninstalling ipykernel-6.30.0:
      Successfully uninstalled ipykernel-6.30.0


ERROR: After October 2020 you may experience errors when installing or updating packages. This is because pip will change the way that it resolves dependency conflicts.

We recommend you use --use-feature=2020-resolver to test your packages with the new resolver before it becomes the default.

notebook 6.4.12 requires ipython-genutils, which is not installed.
notebook 6.4.12 requires Send2Trash>=1.8.0, which is not installed.
nbclassic 0.4.4 requires ipython-genutils, which is not installed.
nbclassic 0.4.4 requires Send2Trash>=1.8.0, which is not installed.
You should consider upgrading via the 'c:\users\sezar\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.
